# Round 3 — Pairwise Diebold-Mariano comparison across all baselines (Issue #50)

Compares **Naive, ARIMA-AIC, ARIMA-BIC, VAR-AIC, VAR-BIC, VECM (6-var), and LSTM** pairwise at
1-/5-/20-day horizons, per proposal Section 5.4/Section 7. This is the pairwise-significance layer on top of
each baseline's own vs-naive comparison (#54, #28, #29, #47, #49) -- the question here isn't just
"does each model beat naive," it's "which of these differences are statistically real, and which
are noise."

**ARIMA has two order selections (AIC vs BIC), VAR has two lag-order selections (AIC vs BIC) --
both kept as separate arms rather than picking a winner, per the Round 3 sensitivity-check
precedent in `M2_CHECKLIST.md` (Aug 16 decision). VECM is scored on its 6-variable system, to
match ARIMA/VAR-AIC/VAR-BIC/LSTM's proposal-correct feature set (Aug 19 usdcad decision) -- this
is also `johansen_vecm.ipynb`'s primary system as of the Aug 21 relabeling (`M2_CHECKLIST.md`),
which corrected #47 to match the Aug 19 decision like #28/#29/#49 were.**

## The data-alignment problem this notebook used to have to work around (fixed by issue #63)

ARIMA/VAR-AIC/VAR-BIC used to load data through `src/EDA_VAR_AIC _lag _order.py`'s `load_levels()`
(inner join across BoC/FRED/CPI, 698 origins) -- a different, unreconciled pipeline from VECM/LSTM's
`src/gold_feature_pipeline.py`'s `build_gold_features()` (outer join + forward-fill, 4,268-row
calendar). "5 trading days ahead of `origin_date`" meant a different real calendar date depending on
which pipeline produced it, and even the origins that happened to land on the same calendar date
only covered ~19% of possible pairs, mostly by coincidence.

Issue #63 fixed this at the source: ARIMA/VAR-AIC/VAR-BIC now read `data/processed/gold_features.csv`
directly (`load_levels()`, `arima_baseline.ipynb`, `var_bic_baseline.ipynb`), the same calendar
VECM/LSTM already used. That alone wasn't sufficient, though -- it surfaced a second, deeper
disagreement in what "origin" meant: ARIMA/VAR-AIC/VAR-BIC's loops counted an origin by how many
*differenced* observations they'd trained on, while `evaluate_vecm()`'s loop counts it by how many
*level* observations it's trained on -- a permanent one-row phase offset for the same `origin`
number. On the newly-unified calendar that offset actually made direct origin_date overlap *worse*
(0%, since a more regular ffilled calendar removes the holiday irregularities that used to
occasionally break the phase back into sync) until #63 also re-anchored ARIMA/VAR-AIC/VAR-BIC's
loops onto VECM's level-counting convention. Origin_date now matches core/VECM 750/750, every
horizon.

**Cross-pipeline comparisons are still verified by walking each side's own calendar forward
`horizon` positions from `origin_date` and requiring the resulting real target date to match on
both sides** (`src/model_comparison.py`'s `attach_target_date()` / `merge_cross_pipeline()`), rather
than by comparing the recorded `actual` values -- `build_gold_features()` forward-fills yield levels
before the target spread is computed, so ~16% of rows repeat their immediately-prior value and two
genuinely different target dates can coincidentally carry the same `actual` value. Post-#63 this
check is a safety net rather than an active workaround for core/VECM pairs (0 rows ever fail it,
since origin_date now fully determines target_date identically on both sides); LSTM cross-pairs
still see it trim a handful of boundary rows (745 of 750, see the sample-size table below) where
LSTM's own rolling-CV date restriction runs a few origins short of core's range near the end of the
series -- an expected, minor edge effect, not a data-quality problem.

**Result:** every pairwise comparison in this notebook, cross-pipeline or not, now runs on
essentially the full 750-origin sample at every horizon (745-750, see below) -- up from as few as
18 origins for some h=20 cross-pipeline pairs before #63.

## A bug found and fixed while building this (issue #28, pre-#63)

`src/EDA_VAR_AIC _lag _order.py`'s `evaluate()` (issue #28, closed/merged) had an off-by-one: it
anchored each origin's `last_level` one business day past the actual training cutoff, silently
leaking an otherwise-unseen level into both the VAR and naive forecasts, and desyncing #28's origin
grid from every sibling baseline's. Fixed as part of this issue -- see `M2_CHECKLIST.md`'s Aug 20
entry. The substantive finding (VAR does not beat naive at any horizon) is unchanged; the exact
RMSE/MAE/DM figures shifted slightly (significance got *stronger*, not weaker).


In [1]:
import sys
from pathlib import Path

import itertools
import pandas as pd

# Bootstrap guess, just precise enough to import project_paths -- immediately
# replaced below by the authoritative, marker-based find_project_root(), so a
# wrong guess here (e.g. this notebook run from a different working directory)
# doesn't silently propagate into OUT_DIR / the src/ import path.
_bootstrap_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_bootstrap_root / "src"))

from project_paths import find_project_root
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model_comparison import (
    load_core, load_vecm, load_lstm, core_calendar, vecm_calendar, lstm_calendar,
    merge_cross_pipeline, pairwise_dm, plain_language_verdict, HORIZONS, OUT_DIR,
    apply_multiplicity_correction, ALPHA,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Load each baseline's per-origin forecasts and evaluation calendar

`load_core()` merges ARIMA (AIC + BIC), VAR-AIC, and VAR-BIC -- all on `load_levels()`'s 698-origin
grid -- and asserts the merge is exact (no dropped rows), since these three files are expected to
agree bit-for-bit on `origin_date`/`actual`/`naive`.

VECM and LSTM are both restricted to core's date range before any comparison, including their own
vs-naive comparisons -- not just the cross-pipeline pairs -- so every arm in the table is judged
over the same calendar span.

`load_vecm()` reads `outputs/r3_vecm_6var_forecasts.csv` -- `johansen_vecm.ipynb`'s primary
6-variable system (Section 2/6-8), not its 5-variable domestic-only robustness check (§9).

`core_calendar()`/`vecm_calendar()`/`lstm_calendar()` reconstruct each pipeline's own daily index
directly from its loader function (`load_levels()` / `load_gold_levels()` / `load_common_sample()`)
-- this is what `merge_cross_pipeline()` walks forward by `horizon` positions to verify alignment.


In [2]:
core = load_core()
date_min, date_max = core["origin_date"].min(), core["origin_date"].max()

vecm = load_vecm(date_min, date_max)
lstm = load_lstm(date_min, date_max)

cal_core = core_calendar()
cal_vecm = vecm_calendar()
cal_lstm = lstm_calendar()

print(f"core (ARIMA-AIC/BIC, VAR-AIC, VAR-BIC): {len(core)} rows, "
      f"{core['origin_date'].nunique()} unique origins")
print(f"vecm (6-var, restricted to core's span): {len(vecm)} rows, "
      f"{vecm['origin_date'].nunique()} unique origins")
print(f"lstm (restricted to core's span): {len(lstm)} rows, "
      f"{lstm['origin_date'].nunique()} unique origins")


core (ARIMA-AIC/BIC, VAR-AIC, VAR-BIC): 2250 rows, 750 unique origins
vecm (6-var, restricted to core's span): 2250 rows, 750 unique origins
lstm (restricted to core's span): 11175 rows, 3725 unique origins


## 2. Run all 21 pairwise comparisons at each horizon

Arms: Naive, ARIMA-AIC, ARIMA-BIC, VAR-AIC, VAR-BIC, VECM, LSTM -- C(7,2) = 21 pairs x 3 horizons =
63 DM tests.

- **Within-core pairs** (Naive + the four load_levels()-based baselines): exact-aligned already, no
  cross-pipeline verification needed.
- **Naive vs VECM / Naive vs LSTM**: each is a single-file comparison (naive and the model's own
  forecast live in the same row), so no cross-pipeline join is needed either.
- **VECM/LSTM vs the four core baselines, and VECM vs LSTM**: genuinely cross-pipeline, verified via
  `merge_cross_pipeline()`'s target-date check.

`pairwise_dm()` reports `insufficient_sample=True` instead of raising when a pair's surviving
sample is too small for the requested horizon (`dieboldmariano` requires `n > h`) -- several
cross-pipeline h=20 pairs land right at that edge once target-date verification removes the
false-tie rows a value-based check would have kept.


In [3]:
CORE_MODELS = ["naive", "arima_aic", "arima_bic", "var_aic", "var_bic"]
results = []

# -- Within-core pairs (10) --
for a, b in itertools.combinations(CORE_MODELS, 2):
    for h in HORIZONS:
        sub = core[core.horizon == h]
        row = pairwise_dm(sub["actual"].values, sub[a].values, sub[b].values, h)
        row.update({"horizon": h, "model_a": a, "model_b": b, "pipeline_pair": "same"})
        row["verdict"] = plain_language_verdict(row, a, b)
        results.append(row)

# -- Naive vs VECM, Naive vs LSTM (2, single-file) --
for name, df, col in [("vecm", vecm, "vecm"), ("lstm", lstm, "lstm")]:
    for h in HORIZONS:
        sub = df[df.horizon == h]
        row = pairwise_dm(sub["actual"].values, sub["naive"].values, sub[col].values, h)
        row.update({"horizon": h, "model_a": "naive", "model_b": name, "pipeline_pair": "same"})
        row["verdict"] = plain_language_verdict(row, "naive", name)
        results.append(row)

# -- VECM/LSTM vs the 4 core baselines (8, cross-pipeline) --
for name, df, col, cal in [("vecm", vecm, "vecm", cal_vecm), ("lstm", lstm, "lstm", cal_lstm)]:
    for cm in ["arima_aic", "arima_bic", "var_aic", "var_bic"]:
        merged = merge_cross_pipeline(
            core[["origin_date", "horizon", "actual", "naive", cm]], cm, cal_core,
            df, col, cal,
        )
        for h in HORIZONS:
            sub = merged[merged.horizon == h]
            row = pairwise_dm(sub["actual"].values, sub[cm].values, sub[col].values, h)
            row.update({"horizon": h, "model_a": cm, "model_b": name, "pipeline_pair": "cross"})
            row["verdict"] = plain_language_verdict(row, cm, name)
            results.append(row)

# -- VECM vs LSTM (1, cross-pipeline) --
merged = merge_cross_pipeline(vecm, "vecm", cal_vecm, lstm, "lstm", cal_lstm)
for h in HORIZONS:
    sub = merged[merged.horizon == h]
    row = pairwise_dm(sub["actual"].values, sub["vecm"].values, sub["lstm"].values, h)
    row.update({"horizon": h, "model_a": "vecm", "model_b": "lstm", "pipeline_pair": "cross"})
    row["verdict"] = plain_language_verdict(row, "vecm", "lstm")
    results.append(row)

dm_all = pd.DataFrame(results)
cols = ["horizon", "model_a", "model_b", "pipeline_pair", "n_forecasts", "insufficient_sample",
        "dm_stat_squared_loss", "dm_p_value_squared_loss",
        "dm_stat_absolute_loss", "dm_p_value_absolute_loss",
        "a_significantly_better_rmse", "b_significantly_better_rmse",
        "a_significantly_better_mae", "b_significantly_better_mae", "verdict"]
dm_all = dm_all[cols].sort_values(["horizon", "model_a", "model_b"]).reset_index(drop=True)

assert len(dm_all) == 63, f"expected 63 rows (21 pairs x 3 horizons), got {len(dm_all)}"
assert dm_all.groupby(["model_a", "model_b"]).ngroups == 21, "expected 21 unique pairs"

# Benjamini-Hochberg FDR correction (issue #69) -- added as extra `_bh` columns
# alongside the raw ones, in the same file, rather than a second CSV. See
# section 3.5 below for why this matters at 63 simultaneous tests.
dm_all = apply_multiplicity_correction(dm_all, alpha=ALPHA)

dm_all.to_csv(OUT_DIR / "r3_pairwise_diebold_mariano.csv", index=False)
print(f"Saved {len(dm_all)} rows (raw + BH-adjusted columns) to outputs/r3_pairwise_diebold_mariano.csv")
dm_all


Saved 63 rows (raw + BH-adjusted columns) to outputs/r3_pairwise_diebold_mariano.csv


,horizon,model_a,model_b,pipeline_pair,n_forecasts,insufficient_sample,dm_stat_squared_loss,dm_p_value_squared_loss,dm_stat_absolute_loss,dm_p_value_absolute_loss,a_significantly_better_rmse,b_significantly_better_rmse,a_significantly_better_mae,b_significantly_better_mae,verdict,dm_p_adj_squared_loss,dm_p_adj_absolute_loss,a_significantly_better_rmse_bh,b_significantly_better_rmse_bh,a_significantly_better_mae_bh,b_significantly_better_mae_bh,verdict_bh
0,1,arima_aic,arima_bic,same,750,False,1.266,0.2060,1.736,0.0830,False,False,False,False,no significant difference on either RMSE or MAE,0.393273,0.177416,False,False,False,False,no significant difference on either RMSE or MAE
1,1,arima_aic,lstm,cross,745,False,1.160,0.2465,0.497,0.6197,False,False,False,False,no significant difference on either RMSE or MAE,0.398192,0.697163,False,False,False,False,no significant difference on either RMSE or MAE
2,1,arima_aic,var_aic,same,750,False,-2.481,0.0133,-3.197,0.0014,True,False,True,False,arima_aic significantly better than var_aic (R...,0.084323,0.008820,False,False,True,False,arima_aic significantly better than the rival ...
3,1,arima_aic,var_bic,same,750,False,1.266,0.2058,1.742,0.0819,False,False,False,False,no significant difference on either RMSE or MAE,0.393273,0.177416,False,False,False,False,no significant difference on either RMSE or MAE
4,1,arima_aic,vecm,cross,750,False,-2.152,0.0317,-2.872,0.0042,True,False,True,False,arima_aic significantly better than vecm (RMSE...,0.126787,0.023625,False,False,True,False,arima_aic significantly better than the rival ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,20,var_aic,var_bic,same,750,False,-0.455,0.6493,0.074,0.9411,False,False,False,False,no significant difference on either RMSE or MAE,0.772879,0.946700,False,False,False,False,no significant difference on either RMSE or MAE
59,20,var_aic,vecm,cross,750,False,0.806,0.4205,1.208,0.2273,False,False,False,False,no significant difference on either RMSE or MAE,0.587407,0.376839,False,False,False,False,no significant difference on either RMSE or MAE
60,20,var_bic,lstm,cross,745,False,0.728,0.4670,1.095,0.2739,False,False,False,False,no significant difference on either RMSE or MAE,0.600429,0.403933,False,False,False,False,no significant difference on either RMSE or MAE
61,20,var_bic,vecm,cross,750,False,0.888,0.3751,0.951,0.3419,False,False,False,False,no significant difference on either RMSE or MAE,0.537075,0.468254,False,False,False,False,no significant difference on either RMSE or MAE


## 3. Sample sizes by pair type

Cross-pipeline pairs have a materially smaller, non-constant `n_forecasts` than same-pipeline pairs
-- worth surfacing explicitly rather than letting it hide inside a single aggregate table. At h=20
some cross-pipeline pairs (vs. VECM) drop as low as 18 paired origins -- below the threshold
`pairwise_dm()` needs to run the test at all, and reported as `insufficient_sample` rather than a
silent or crashing result.


In [4]:
dm_all.groupby(["horizon", "pipeline_pair"])["n_forecasts"].agg(["min", "median", "max", "count"])


min  median   max  count
horizon pipeline_pair                          
1       cross          745   745.0   750      9
        same           750   750.0  3725     12
5       cross          745   745.0   750      9
        same           750   750.0  3725     12
20      cross          745   745.0   750      9
        same           750   750.0  3725     12

## 3.5. Multiple-testing correction (issue #69)

63 simultaneous DM tests (21 pairs x 3 horizons) at an unadjusted alpha=0.05 gives a family-wise
error rate of 1-(1-0.05)**63 ~= 96% -- at that rate, at least one "significant" result is close to
guaranteed by chance alone even if no real pairwise differences exist anywhere in the table. The
raw p-values above are still reported (nothing is deleted), but they shouldn't be read at face
value on their own.

`apply_multiplicity_correction()` (`src/model_comparison.py`) applies Benjamini-Hochberg FDR
control separately to the 63 squared-loss p-values and the 63 absolute-loss p-values (RMSE and MAE
are already treated as independent tracks everywhere else in this module), adding `_bh` columns
alongside the raw ones in the same `outputs/r3_pairwise_diebold_mariano.csv`.

In [5]:
raw_sig = (
    dm_all["a_significantly_better_rmse"] | dm_all["b_significantly_better_rmse"]
    | dm_all["a_significantly_better_mae"] | dm_all["b_significantly_better_mae"]
)
bh_sig = (
    dm_all["a_significantly_better_rmse_bh"] | dm_all["b_significantly_better_rmse_bh"]
    | dm_all["a_significantly_better_mae_bh"] | dm_all["b_significantly_better_mae_bh"]
)
print(f"Raw significant (unadjusted, alpha={ALPHA}): {raw_sig.sum()} of 63")
print(f"BH-adjusted significant: {bh_sig.sum()} of 63")
print()
print("Which raw-significant pairs survive BH correction, by horizon:")
print(pd.DataFrame({"raw_significant": raw_sig, "bh_significant": bh_sig, "horizon": dm_all["horizon"]})
      .groupby("horizon")[["raw_significant", "bh_significant"]].sum())

Raw significant (unadjusted, alpha=0.05): 26 of 63
BH-adjusted significant: 16 of 63

Which raw-significant pairs survive BH correction, by horizon:
         raw_significant  bh_significant
horizon                                 
1                     15              12
5                     11               4
20                     0               0


## 4. Significant results only, in plain language

Per issue #50's Definition of Done: state in plain language which gains are statistically
significant vs. which are noise, at all three horizons. **Reports BH-adjusted significance
(`*_bh` columns, section 3.5), not the raw unadjusted flags** -- per issue #69, so this section
matches the confidence level the numbers actually support. Filtered on the underlying boolean
significance flags (saved alongside the verdict text in the CSV), not by string-matching the
verdict sentence -- so this stays correct even if the wording of a "not significant" verdict
changes later.

In [6]:
is_significant_bh = (
    dm_all["a_significantly_better_rmse_bh"] | dm_all["b_significantly_better_rmse_bh"]
    | dm_all["a_significantly_better_mae_bh"] | dm_all["b_significantly_better_mae_bh"]
)
significant = dm_all[is_significant_bh]
insufficient = dm_all[dm_all["insufficient_sample"]]

for h in HORIZONS:
    print(f"\n=== Horizon {h} day(s) ===")
    sub = significant[significant.horizon == h]
    if sub.empty:
        print("  No BH-significant pairwise differences at this horizon.")
    for _, r in sub.iterrows():
        print(f"  {r.model_a} vs {r.model_b} (n={r.n_forecasts}): {r.verdict_bh}")
    thin = insufficient[insufficient.horizon == h]
    for _, r in thin.iterrows():
        print(f"  {r.model_a} vs {r.model_b}: {r.verdict_bh}")


=== Horizon 1 day(s) ===
  arima_aic vs var_aic (n=750): arima_aic significantly better than the rival on MAE only (p=0.0088); RMSE not significant
  arima_aic vs vecm (n=750): arima_aic significantly better than the rival on MAE only (p=0.0236); RMSE not significant
  arima_bic vs var_aic (n=750): arima_bic significantly better than the rival on MAE only (p=0.0032); RMSE not significant
  arima_bic vs var_bic (n=750): var_bic significantly better than the rival on MAE only (p=0.0236); RMSE not significant
  arima_bic vs vecm (n=750): arima_bic significantly better than the rival on MAE only (p=0.0063); RMSE not significant
  naive vs lstm (n=3725): naive significantly better than the rival on MAE only (p=0.0000); RMSE not significant
  naive vs var_aic (n=750): naive significantly better than the rival on MAE only (p=0.0032); RMSE not significant
  naive vs vecm (n=750): naive significantly better than the rival on MAE only (p=0.0038); RMSE not significant
  var_aic vs lstm (n=745): 

## 5. Summary

**After Benjamini-Hochberg correction (issue #69), 16 of the 26 raw-significant pairs survive --
12 at h=1, 4 at h=5, none at h=20.** The correction narrows how much of the raw table can be
trusted, but does not overturn the substantive finding:

- Of the 16 BH-significant pairs, **Naive is on the winning side in 4** (beats `lstm`, `var_aic`,
  and `vecm` at h=1, and `var_aic` again at h=5) **and on the losing side in 0** -- no model
  significantly beats Naive at any BH-adjusted-significant horizon.
- **Every one of the 16 BH-significant results is significant on MAE only** -- none survive under
  squared-error (RMSE) loss, at either horizon.
- The other 12 BH-significant pairs are comparisons among non-naive models. `var_aic` is on the
  losing side in 7 of them (beaten by `arima_aic`, `arima_bic`, `lstm`, and `var_bic`, across both
  horizons); `vecm` loses the other 4 it's involved in (to `arima_aic`, `arima_bic`, `var_bic`, and
  `lstm`, all at h=1); `arima_bic` loses once, to `var_bic` (h=1).
- h=20 keeps its zero-significant-results character from the raw table straight through BH
  correction; h=5 now clears the BH bar for 4 of its 11 raw-significant pairs -- weaker than h=1,
  but no longer wiped out entirely the way it was on the pre-#63 sample.

**Bottom line: correcting for 63 simultaneous tests, now on the full reconciled-calendar sample,
still makes the evidence thinner, not different in direction** -- the "Naive is never
significantly beaten" headline survives, now resting on a broader set of results (16 vs. the
pre-#63 sample's 9) spanning both h=1 and h=5 rather than h=1 alone.

## Calendar reconciliation (issue #63) -- resolved

The cross-pipeline sample-size reduction previously described here (as few as 18 origins for some
h=20 pairs, driven by `load_levels()` and `build_gold_features()` never having been reconciled onto
one shared calendar) is fixed as of issue #63: every pairwise comparison above runs on 745-750 of a
possible 750 origins at every horizon (see section 3's table), and `insufficient_sample` is 0 for
all 63 rows -- none of Round 3's headline findings were ever resting on a thin cross-pipeline
sample, but the underlying comparisons -- and the multiple-testing correction above -- are
meaningfully more complete now.